# NB_hospital_daily_summary

Builds `adb_qnxthealth.gold.hospital_daily_summary` — one row per hospital per calendar day with test volumes, patient/doctor counts, abnormal/critical rates, and day-over-day trend via LAG(). Full rebuild every run.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_qnxthealth.gold.hospital_daily_summary AS

WITH daily_agg AS (
  SELECT
    lr.hospital_id,
    lr.test_date                                                         AS summary_date,
    COUNT(*)                                                             AS total_tests,
    COUNT(DISTINCT lr.patient_id)                                        AS unique_patients_tested,
    COUNT(DISTINCT lr.doctor_id)                                         AS active_doctor_count,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)           AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)           AS critical_test_count
  FROM adb_qnxthealth.silver.lab_results lr
  GROUP BY lr.hospital_id, lr.test_date
),

with_rates AS (
  SELECT
    da.*,
    CAST(ROUND(da.abnormal_test_count * 100.0 / da.total_tests, 2) AS DECIMAL(5,2))  AS abnormal_rate_pct,
    CAST(ROUND(da.critical_test_count * 100.0 / da.total_tests, 2) AS DECIMAL(5,2))  AS critical_rate_pct,
    LAG(da.total_tests) OVER (PARTITION BY da.hospital_id ORDER BY da.summary_date)   AS prev_day_total_tests
  FROM daily_agg da
)

SELECT
  wr.hospital_id,
  h.hospital_name,
  wr.summary_date,
  c.day_name,
  c.is_weekend,
  wr.total_tests,
  wr.unique_patients_tested,
  wr.active_doctor_count,
  wr.abnormal_test_count,
  wr.critical_test_count,
  wr.abnormal_rate_pct,
  wr.critical_rate_pct,
  wr.prev_day_total_tests,
  CAST(
    CASE
      WHEN wr.prev_day_total_tests IS NULL OR wr.prev_day_total_tests = 0 THEN NULL
      ELSE ROUND((wr.total_tests - wr.prev_day_total_tests) * 100.0 / wr.prev_day_total_tests, 2)
    END AS DECIMAL(6,2)
  )                                                                      AS test_volume_change_pct

FROM      with_rates                              wr
LEFT JOIN adb_qnxthealth.silver.hospitals          h  ON wr.hospital_id = h.hospital_id
LEFT JOIN adb_qnxthealth.silver.calendar           c  ON c.date_key = CAST(date_format(wr.summary_date, 'yyyyMMdd') AS INT)
ORDER BY  wr.hospital_id, wr.summary_date;

In [0]:
%skip
%sql
SELECT count(*) AS total_rows,
       count(DISTINCT hospital_id) AS unique_hospitals,
       min(summary_date) AS min_date,
       max(summary_date) AS max_date,
       avg(total_tests) AS avg_daily_tests,
       count_if(prev_day_total_tests IS NOT NULL) AS rows_with_trend
FROM   adb_qnxthealth.gold.hospital_daily_summary;

In [0]:
record_count = spark.table("adb_qnxthealth.gold.hospital_daily_summary").count()
dbutils.notebook.exit(str(record_count))